# 🎾 Capítulo 3 — Los Herederos del Trono
### Sinner vs Alcaraz: ¿Quién domina la nueva era? (2015-2024)
**Fuente de datos:** [Jeff Sackmann — tennis_atp](https://github.com/JeffSackmann/tennis_atp)  
**Torneos analizados:** Grand Slam y Masters 1000 (2015-2024)

---
## 1. Imports y Carga de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

# Cargar datos 2015-2024
anos = list(range(2015, 2025))
dfs = []

for ano in anos:
    url = f"https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_{ano}.csv"
    df_ano = pd.read_csv(url)
    df_ano['season'] = ano
    dfs.append(df_ano)
    print(f"✅ {ano} cargado — {df_ano.shape[0]} partidos")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal partidos: {df.shape[0]}")
print(f"Total columnas: {df.shape[1]}")

---
## 2. Definición de Jugadores y Filtros

In [ ]:
sinner_alcaraz = ['Jannik Sinner', 'Carlos Alcaraz']

big3 = ['Roger Federer', 'Rafael Nadal', 'Novak Djokovic']
gen_olvidada = ['Alexander Zverev', 'Daniil Medvedev',
                'Dominic Thiem', 'Stefanos Tsitsipas']

todos_anos = list(range(2015, 2025))

colores_sa = {
    'Jannik Sinner':  '#1a78cf',
    'Carlos Alcaraz': '#e8462a'
}

def clasificar_jugador(nombre):
    if nombre in sinner_alcaraz:
        return nombre
    elif nombre in big3:
        return 'Big 3'
    elif nombre in gen_olvidada:
        return 'Gen Olvidada'
    else:
        return 'Resto'

# Filtrar Slams y Masters 1000
df_grandes = df[df['tourney_level'].isin(['G', 'M'])].copy()

# Solo finales
finales = df_grandes[df_grandes['round'] == 'F'].copy()

# Títulos individuales por año
titulos_sa = finales[
    finales['winner_name'].isin(sinner_alcaraz)
].groupby(['season', 'winner_name']).size().reset_index(name='titulos')

print("Torneos ganados por Sinner y Alcaraz:")
print(finales[
    finales['winner_name'].isin(sinner_alcaraz)
][['season', 'tourney_name', 'surface', 'winner_name']].sort_values('season').to_string(index=False))

---
## 3. Visualización 1 — Títulos por Año

Alcaraz llegó primero y más joven al alto nivel en 2022. Sinner arrancó más tarde pero en 2024 lo superó con 5 títulos grandes, una cifra histórica para su generación.

In [ ]:
# Rellenar años sin títulos con cero
datos_completos = []
for jugador in sinner_alcaraz:
    for ano in todos_anos:
        dato = titulos_sa[
            (titulos_sa['winner_name'] == jugador) &
            (titulos_sa['season'] == ano)
        ]['titulos'].values
        datos_completos.append({
            'season': ano,
            'winner_name': jugador,
            'titulos': dato[0] if len(dato) > 0 else 0
        })

df_sa = pd.DataFrame(datos_completos)

fig, ax = plt.subplots(figsize=(14, 7))

for jugador in sinner_alcaraz:
    datos = df_sa[df_sa['winner_name'] == jugador]
    ax.plot(datos['season'], datos['titulos'],
            marker='o', linewidth=2.5, markersize=8,
            label=jugador, color=colores_sa[jugador])
    ax.fill_between(datos['season'], datos['titulos'],
                    alpha=0.15, color=colores_sa[jugador])
    for _, row in datos.iterrows():
        if row['titulos'] > 0:
            ax.annotate(f"{int(row['titulos'])}",
                       (row['season'], row['titulos']),
                       textcoords="offset points",
                       xytext=(0, 10), ha='center', fontsize=10,
                       fontweight='bold', color=colores_sa[jugador])

ax.set_title('Los Herederos del Trono\nTítulos en Slams y Masters 1000 (2015-2024)',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Temporada', fontsize=12)
ax.set_ylabel('Títulos', fontsize=12)
ax.set_xticks(todos_anos)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('08_sinner_vs_alcaraz_titulos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Visualización 2 — Distribución por Superficie

El contraste más revelador: Sinner es un especialista puro en cancha dura (100%), mientras Alcaraz gana en todas las superficies, convirtiéndose en el jugador más completo de su generación.

In [ ]:
titulos_superficie_sa = finales[
    finales['winner_name'].isin(sinner_alcaraz)
].groupby(['winner_name', 'surface']).size().reset_index(name='titulos')

superficies = ['Hard', 'Clay', 'Grass']
colores_superficie = {
    'Hard':  '#4878cf',
    'Clay':  '#c45b49',
    'Grass': '#2ca02c'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, jugador in enumerate(sinner_alcaraz):
    ax = axes[idx]
    datos = titulos_superficie_sa[titulos_superficie_sa['winner_name'] == jugador]

    valores = []
    etiquetas = []
    colores_pie = []

    for sup in superficies:
        dato = datos[datos['surface'] == sup]['titulos'].values
        if len(dato) > 0 and dato[0] > 0:
            valores.append(dato[0])
            etiquetas.append(sup)
            colores_pie.append(colores_superficie[sup])

    wedges, texts, autotexts = ax.pie(
        valores,
        labels=etiquetas,
        colors=colores_pie,
        autopct='%1.0f%%',
        startangle=90,
        textprops={'fontsize': 12}
    )

    for autotext in autotexts:
        autotext.set_fontweight('bold')
        autotext.set_fontsize(13)

    ax.set_title(jugador, fontsize=14, fontweight='bold',
                 color=colores_sa[jugador], pad=15)

fig.suptitle('Distribución de títulos por superficie\nSlams y Masters 1000 (2015-2024)',
             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('09_superficie_sinner_alcaraz.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Visualización 3 — Enfrentamientos Directos (H2H)

7 duelos en torneos grandes entre 2021 y 2024. Alcaraz lidera 5-2 y ha ganado en todas las superficies. Sinner solo le ha ganado en dura y césped.

In [ ]:
# Buscar partidos donde se enfrentaron
h2h = df_grandes[
    (df_grandes['winner_name'].isin(sinner_alcaraz)) &
    (df_grandes['loser_name'].isin(sinner_alcaraz))
].copy()

h2h['ganador']   = h2h['winner_name']
h2h['perdedor']  = h2h['loser_name']

print(f"Total enfrentamientos: {len(h2h)}")
print(h2h[['season', 'tourney_name', 'round', 'surface', 'ganador']].to_string(index=False))

victorias_h2h    = h2h.groupby('ganador').size().reset_index(name='victorias')
h2h_superficie   = h2h.groupby(['ganador', 'surface']).size().reset_index(name='victorias')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# H2H total
jugadores_h2h = ['Carlos Alcaraz', 'Jannik Sinner']
wins = [
    victorias_h2h[victorias_h2h['ganador'] == j]['victorias'].values
    for j in jugadores_h2h
]
wins = [w[0] if len(w) > 0 else 0 for w in wins]

bars = ax1.bar(jugadores_h2h, wins,
               color=[colores_sa[j] for j in jugadores_h2h],
               alpha=0.85, width=0.5)

for bar, val in zip(bars, wins):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(val), ha='center', fontsize=20, fontweight='bold')

ax1.set_title('Head to Head Total\nTorneos Grandes (2021-2024)',
              fontsize=13, fontweight='bold')
ax1.set_ylabel('Victorias', fontsize=11)
ax1.set_ylim(0, 7)
ax1.grid(True, alpha=0.3, axis='y')

# H2H por superficie
superficies_h2h = ['Hard', 'Clay', 'Grass']
colores_sup = {
    'Hard':  '#4878cf',
    'Clay':  '#c45b49',
    'Grass': '#2ca02c'
}

x = range(len(jugadores_h2h))
ancho = 0.25

for i, sup in enumerate(superficies_h2h):
    valores = []
    for jugador in jugadores_h2h:
        dato = h2h_superficie[
            (h2h_superficie['ganador'] == jugador) &
            (h2h_superficie['surface'] == sup)
        ]['victorias'].values
        valores.append(dato[0] if len(dato) > 0 else 0)
    ax2.bar([p + i * ancho for p in x], valores,
            width=ancho, label=sup,
            color=colores_sup[sup], alpha=0.85)

ax2.set_title('Head to Head por Superficie\nTorneos Grandes (2021-2024)',
              fontsize=13, fontweight='bold')
ax2.set_ylabel('Victorias', fontsize=11)
ax2.set_xticks([p + ancho for p in x])
ax2.set_xticklabels(jugadores_h2h, fontsize=11)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

fig.suptitle('Alcaraz vs Sinner — Enfrentamientos Directos\nTorneos Grandes (2021-2024)',
             fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('10_h2h_alcaraz_sinner.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Conclusiones

El análisis de Sinner y Alcaraz como herederos del trono revela una rivalidad fascinante entre dos estilos completamente distintos:

- **Alcaraz** llegó primero y más joven al alto nivel, ganando 3 títulos grandes en 2022 con apenas 19 años.
- **Sinner** explotó en 2024 con 5 títulos grandes, una cifra que solo el Big 3 había logrado antes.
- **Alcaraz** es el jugador más completo: gana en cancha dura (44%), tierra (33%) y césped (22%).
- **Sinner** es un especialista puro en cancha dura, con el 100% de sus títulos grandes en esa superficie.
- En duelos directos en torneos grandes, **Alcaraz lidera 5-2**, ganando en todas las superficies.
- La pregunta que queda abierta: ¿podrá Sinner conquistar tierra y césped, o Alcaraz consolidará su ventaja como el jugador más completo?

> *"Sinner gana más, Alcaraz gana en todas partes. La rivalidad que definirá el tenis de la próxima década apenas comienza."*